In [57]:
import os
from dotenv import load_dotenv

# Load API keys from .env
load_dotenv()
if os.getenv("GROQ_API_KEY"):
    os.environ["Groq_API_KEY"] = os.environ["GROQ_API_KEY"]

CROSSENCODER_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
EMBEDDING_MODEL = "BAAI/bge-m3"


In [58]:
from utils import chunking_embedding

retriever,vectorstore,store = chunking_embedding(embedding_model=EMBEDDING_MODEL, loaded_files=None, is_for_embedding=False)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [59]:
import json
with open("./golden_dataset.json","r") as f:
    golden_dataset = json.load(f)

### advaced retreiver

In [60]:
# =========================================================================
# CELL 2: Initialize BM25 and Cross-Encoder
# =========================================================================
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from sentence_transformers import CrossEncoder

# 1. Extract all child chunks from the Chroma vectorstore to initialize BM25
chroma_data = vectorstore.get()
child_docs = [
    Document(page_content=content, metadata=metadata)
    for content, metadata in zip(chroma_data['documents'], chroma_data['metadatas'])
]

# 2. Initialize the Sparse Retriever (BM25)
bm25_retriever = BM25Retriever.from_documents(child_docs)

# 3. Initialize the Cross-Encoder (Reranker)
# This model is small, fast, and highly effective for reranking
reranker = CrossEncoder(CROSSENCODER_MODEL)
# reranker = CrossEncoder(
#     "BAAI/bge-reranker-v2-m3",
#     max_length=1024,   # bge-reranker-v2-m3 supports longer inputs than the mini English model
# )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [61]:
# =========================================================================
# CELL 3: Define the Advanced Retrieval Pipeline with Metadata Filtering
# =========================================================================
SPORT_FILE_MAP = {
    "football": "Football.pdf",
    "basketball": "Basketball.pdf",
    "handball": "Handball.pdf",
    "tennis": "Tennis.pdf",
    "boxing": "Boxing.pdf"
}

def map_sport_to_filename(sport: str | None) -> str | None:
    """Maps sport name to disk file name (e.g. 'football' -> 'Football.pdf')."""
    if not sport:
        return None
    cleaned = str(sport).strip().lower().replace(".pdf", "")
    return SPORT_FILE_MAP.get(cleaned)

def advanced_parent_child_retrieval(query: str, top_k_children: int = 25, top_n_rerank: int = 5, sport: str | None = None):
    # Map sport to disk file name (e.g. football -> Football.pdf)
    target_pdf = map_sport_to_filename(sport)
    target_file_name = target_pdf.replace(".pdf", "") if target_pdf else None
    
    # ---------------------------------------------------------
    # [1. Hybrid Search on Child Chunks with Sport Filter]
    # ---------------------------------------------------------
    if target_file_name:
        dense_children = vectorstore.similarity_search(
            query, 
            k=top_k_children, 
            filter={"file_name": target_file_name}
        )
    else:
        dense_children = vectorstore.similarity_search(query, k=top_k_children)
    
    bm25_retriever.k = top_k_children * 2 if target_file_name else top_k_children
    all_sparse = bm25_retriever.invoke(query)
    if target_file_name:
        sparse_children = [
            d for d in all_sparse 
            if d.metadata.get("file_name") == target_file_name
        ][:top_k_children]
    else:
        sparse_children = all_sparse
    
    # ---------------------------------------------------------
    # [2. Reciprocal Rank Fusion (RRF)]
    # ---------------------------------------------------------
    fused_scores = {}
    unique_children_map = {}
    
    for rank, doc in enumerate(dense_children):
        content = doc.page_content
        fused_scores[content] = fused_scores.get(content, 0) + (1 / (rank + 60))
        unique_children_map[content] = doc

    for rank, doc in enumerate(sparse_children):
        content = doc.page_content
        fused_scores[content] = fused_scores.get(content, 0) + (1 / (rank + 60))
        unique_children_map[content] = doc
        
    # Sort and take top 30 unique child candidates
    fused_candidates = sorted(
        unique_children_map.values(), 
        key=lambda d: fused_scores[d.page_content], 
        reverse=True
    )[:30]
    
    # ---------------------------------------------------------
    # [3. Cross-Encoder Reranking]
    # ---------------------------------------------------------
    cross_encoder_inputs = [[query, doc.page_content] for doc in fused_candidates]
    rerank_scores = reranker.predict(cross_encoder_inputs)
    
    for doc, score in zip(fused_candidates, rerank_scores):
        doc.metadata['rerank_score'] = score
        
    top_n_children = sorted(
        fused_candidates, 
        key=lambda d: d.metadata['rerank_score'], 
        reverse=True
    )[:top_n_rerank]
    
    # ---------------------------------------------------------
    # [4. Parent Document Resolution]
    # ---------------------------------------------------------
    unique_parent_ids = []
    seen_ids = set()
    
    for child in top_n_children:
        parent_id = child.metadata.get("doc_id") 
        if parent_id and parent_id not in seen_ids:
            seen_ids.add(parent_id)
            unique_parent_ids.append(parent_id)
            
    resolved_parent_docs = store.mget(unique_parent_ids)
    
    final_context = []
    for parent_id, doc in zip(unique_parent_ids, resolved_parent_docs):
        if doc is not None:
            doc.metadata["doc_id"] = parent_id
            final_context.append(doc)
            
    return final_context


In [62]:
# # add normal retriever context
# print("Starting retrieval process...")

# for item in golden_dataset:
#     # Query the retriever using the specific question
#     normal_retrieved_docs = retriever.invoke(item["question"])
#     advanced_retrieved_docs = advanced_parent_child_retrieval(item["question"])
    
#     # Extract the text content from each retrieved document and assign it to the context list
#     item["normal_retriever_context"] = normal_retrieved_docs
#     item["advanced_retriever_context"] = advanced_retrieved_docs

#     # Print progress to the terminal
#     print(f"Processed {item['question_id']}: Retrieved {len(normal_retrieved_docs)} normal parent chunks and {len(advanced_retrieved_docs)} advanced parent chunks.")

# print("\nRetrieval complete! Dataset is ready for evaluation.")

In [63]:
# import os
# from langchain_core.prompts import PromptTemplate
# from langchain_groq import ChatGroq
# from langchain_core.output_parsers import StrOutputParser
# from langchain_ollama import ChatOllama


# # 1. Initialize the Groq LLM
# # Temperature 0 is critical for evaluation tasks to ensure deterministic, consistent outputs.
# # Llama 3 70B is highly recommended for reasoning and following strict output formats.
# judge_llm = ChatOllama(
#     model="gemma2:2b",
#     temperature=0
# )

# # 2. Define the Strict Prompt Template
# # We explicitly forbid the LLM from adding conversational filler.
# template = """You are an expert Information Retrieval evaluator.
# Your task is to determine if the retrieved context is relevant to the question and contains the correct answer as defined by the ground truth.

# Question: {question}
# Ground Truth: {ground_truth}
# Retrieved Context: {context}

# Evaluate whether the Retrieved Context provides the necessary information to answer the Question accurately. 

# CRITICAL RULE: The Retrieved Context MUST contain the specific facts, details, or information present in the Ground Truth. If the context is only generally related to the question but is missing the actual answer provided in the Ground Truth, you must evaluate it as not relevant.

# Respond ONLY with the exact phrase "relevant" if the context explicitly supports the Ground Truth answer, or "not relevant" if it is missing the necessary information. Do not include any reasoning, explanation, or punctuation."""

# judge_prompt = PromptTemplate(
#     input_variables=["question", "ground_truth", "context"],
#     template=template
# )

# # 3. Build the Evaluation Chain
# # The StrOutputParser() automatically extracts the string from the LLM's message object
# custom_judge = judge_prompt | judge_llm | StrOutputParser()

In [64]:
# all_parent_ids = list(store.yield_keys())
# all_parent_docs = store.mget(all_parent_ids)



# # 3. Process the dataset
# print("Step 2 & 3: Processing the dataset...")
# print("Note: Running the LLM over all database documents will take significant compute time.\n")

# for i, item in enumerate(golden_dataset):
#     target_file_name = item.get("file_name")
#     question = item["question"]
#     ground_truth = item["ground_truth"]
    
#     # --- Step 2: Find all globally relevant docs in the DB ---
#     print(f"[{i+1}/{len(golden_dataset)}] Evaluating entire DB for {item.get('question_id')}...")
#     item["relevant_docs"] = []
    
#     # We can now just iterate directly over the documents since doc_id is in metadata
#     for doc in all_parent_docs:
#         if doc is None:
#             continue
            
#         chunk_file_name = doc.metadata.get("file_name")
        
#         # If filenames do not match, it is automatically not relevant (Skip LLM)
#         if chunk_file_name != target_file_name:
#             continue
            
#         # If filenames match, check with the LLM
#         raw_output = custom_judge.invoke({
#             "question": question,
#             "ground_truth": ground_truth,
#             "context": doc.page_content
#         })
        
#         verdict = raw_output.strip().lower()
        
#         # If the LLM determines it is relevant, append the doc_id
#         if "not relevant" not in verdict and "relevant" in verdict:
#             item["relevant_docs"].append(doc.metadata["doc_id"])
            
#     print(f"   -> Found {len(item['relevant_docs'])} verified relevant docs in the database.")
            
# print("\nEvaluation complete! The golden_dataset now contains globally verified 'relevant_docs' (as IDs).")

In [65]:
# print("Starting evaluation with custom judge...\n")

# for i, item in enumerate(golden_dataset):
#     question = item["question"]
#     ground_truth = item["ground_truth"]
#     normal_contexts = item.get("normal_retriever_context", [])
#     advanced_contexts = item.get("advanced_retriever_context", [])
#     item["normal_relevance"] = []
#     item["advanced_relevance"] = []

    
#     for doc in normal_contexts:
#         if(doc.metadata["doc_id"] in item.get("relevant_docs", [])):
#             item["normal_relevance"].append("relevant")
#         else:
#             item["normal_relevance"].append("not relevant")
#     for doc in advanced_contexts:
#         if(doc.metadata["doc_id"] in item.get("relevant_docs", [])):
#             item["advanced_relevance"].append("relevant")
#         else:
#             item["advanced_relevance"].append("not relevant")
    



In [66]:
# import pickle

# with open("golden_dataset.pkl", "wb") as f:
#     pickle.dump(golden_dataset, f)

In [67]:
# import pickle

# # Open the file in read-binary mode and load the object
# with open('golden_dataset.pkl', 'rb') as file:
#     golden_dataset = pickle.load(file)


In [68]:
# import numpy as np
# import pandas as pd
# from typing import List


# # ---------- Core metric functions ----------

# def get_doc_ids(docs: List) -> List[str]:
#     """Extract doc_id from a list of LangChain Document objects."""
#     return [d.metadata.get("doc_id") for d in docs]


# def precision_at_k(retrieved_ids: List[str], relevant_ids: set, k: int) -> float:
#     if k == 0:
#         return 0.0
#     top_k = retrieved_ids[:k]
#     if len(top_k) == 0:
#         return 0.0
#     hits = sum(1 for doc_id in top_k if doc_id in relevant_ids)
#     return hits / len(top_k)


# def recall_at_k(retrieved_ids: List[str], relevant_ids: set, k: int) -> float:
#     if len(relevant_ids) == 0:
#         return 0.0
#     top_k = retrieved_ids[:k]
#     hits = sum(1 for doc_id in top_k if doc_id in relevant_ids)
#     return hits / len(relevant_ids)


# def reciprocal_rank(retrieved_ids: List[str], relevant_ids: set) -> float:
#     for rank, doc_id in enumerate(retrieved_ids, start=1):
#         if doc_id in relevant_ids:
#             return 1.0 / rank
#     return 0.0


# def average_precision(retrieved_ids: List[str], relevant_ids: set) -> float:
#     if len(relevant_ids) == 0:
#         return 0.0
#     hits = 0
#     precisions = []
#     for rank, doc_id in enumerate(retrieved_ids, start=1):
#         if doc_id in relevant_ids:
#             hits += 1
#             precisions.append(hits / rank)
#     if not precisions:
#         return 0.0
#     return sum(precisions) / len(relevant_ids)


# def dcg_at_k(retrieved_ids: List[str], relevant_ids: set, k: int) -> float:
#     top_k = retrieved_ids[:k]
#     return sum(
#         (1.0 if doc_id in relevant_ids else 0.0) / np.log2(rank + 1)
#         for rank, doc_id in enumerate(top_k, start=1)
#     )


# def ndcg_at_k(retrieved_ids: List[str], relevant_ids: set, k: int) -> float:
#     ideal_hits = min(len(relevant_ids), k)
#     idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, ideal_hits + 1))
#     if idcg == 0:
#         return 0.0
#     return dcg_at_k(retrieved_ids, relevant_ids, k) / idcg


# # ---------- Evaluation over the golden dataset ----------

# def evaluate_retriever(golden_dataset, context_key: str, k_values=(1, 3, 5)):
#     per_query_rows = []

#     for item in golden_dataset:
#         relevant_ids = set(item.get("relevant_docs", []))
#         retrieved_docs = item.get(context_key, [])
#         retrieved_ids = get_doc_ids(retrieved_docs)
#         has_relevant = len(relevant_ids) > 0

#         row = {
#             "question_id": item.get("question_id"),
#             "num_relevant": len(relevant_ids),
#             "num_retrieved": len(retrieved_ids),
#             "MRR": reciprocal_rank(retrieved_ids, relevant_ids) if has_relevant else np.nan,
#             "MAP": average_precision(retrieved_ids, relevant_ids) if has_relevant else np.nan,
#         }

#         for k in k_values:
#             row[f"Precision@{k}"] = precision_at_k(retrieved_ids, relevant_ids, k)
#             row[f"Recall@{k}"] = recall_at_k(retrieved_ids, relevant_ids, k) if has_relevant else np.nan
#             row[f"nDCG@{k}"] = ndcg_at_k(retrieved_ids, relevant_ids, k) if has_relevant else np.nan

#         per_query_rows.append(row)

#     df = pd.DataFrame(per_query_rows)

#     metric_cols = [c for c in df.columns if c not in ("question_id", "num_relevant", "num_retrieved")]
#     # .mean() with skipna=True (default) ignores NaN rows automatically
#     averaged = df[metric_cols].mean(skipna=True).to_dict()

#     return df, averaged


# # ---------- Run for both retrievers ----------

# k_values = (1, 3, 5)

# normal_df, normal_avg = evaluate_retriever(golden_dataset, "normal_retriever_context", k_values)
# advanced_df, advanced_avg = evaluate_retriever(golden_dataset, "advanced_retriever_context", k_values)

# # Side-by-side summary table
# summary = pd.DataFrame({
#     "Normal Retriever": normal_avg,
#     "Advanced Retriever": advanced_avg,
# }).round(4)


# summary.to_csv("evaluation_summary.csv")

# print("=== Per-query results (Normal Retriever) ===")
# display(normal_df)

# print("\n=== Per-query results (Advanced Retriever) ===")
# display(advanced_df)

# print("\n=== Averaged Metrics Comparison ===")
# display(summary)

In [69]:
# advanced_parent_child_retrieval("what's the size of the ball")

### Generation phase

In [70]:
from pydantic import BaseModel, Field
from typing import List

class ClaimSource(BaseModel):
    file_name: str = Field(description="Rulebook name the claim came from, e.g. 'Football', 'Basketball', 'Handball', 'Tennis', 'Boxing'")
    page_number: str = Field(description="Page number in the rulebook where this claim appears")

class Claim(BaseModel):
    claim: str = Field(description="A specific factual statement taken from the context that supports part of the answer")
    source: ClaimSource = Field(description="Exact source (rulebook + page) this claim was extracted from")

class RefereeAnswer(BaseModel):
    answer: str = Field(description="Final answer to the user, written in the same language the user asked in")
    claims: List[Claim] = Field(
        default_factory=list,
        description="Every factual claim backing the answer, each tagged with its source. MUST be empty if the answer is a greeting/chitchat, or if no rule was found."
    )

In [71]:
from langchain_core.prompts import ChatPromptTemplate

sports_reference_system_prompt = """<role>
You are an expert multi-sport rules referee assistant. You have access to the official rulebooks for five sports: Football, Basketball, Handball, Tennis, and Boxing. Your job is to give referees, players, and fans accurate, rulebook-grounded answers to their questions — never guesses, never outside knowledge.
</role>

<chat_history>
{chat_history}
</chat_history>

<context>
Below are excerpts retrieved from the rulebooks relevant to the user's question. Each excerpt is tagged with its source rulebook name and page number, in this format:

[Rulebook Name | Page X]:
<excerpt text>

Excerpts are separated by "---". There may be excerpts from more than one sport if the retrieval system matched multiple rulebooks — only use the ones that are actually relevant to the question asked.

RETRIEVED CONTEXT:
{context}
</context>

<instructions>
1. LANGUAGE: Always answer in the same language the user used in their question. If they ask in Arabic, answer in Arabic. If they ask in English, answer in English. Never switch languages unless the user does.

2. GREETINGS & CHITCHAT: If the user's message is a greeting, thanks, small talk, or anything not actually a rules question (e.g. "hello", "thank you", "who are you?", "مرحبا", "شكرا"), respond naturally and warmly WITHOUT searching the context or citing any claims. Leave the "claims" list empty in this case.

3. RULE QUESTIONS — GROUNDED ANSWERS ONLY: For any actual rules question, your answer must be based ONLY on the RETRIEVED CONTEXT above. Do not use any outside knowledge of sports rules, even if you're confident it's correct. The rulebooks are the single source of truth, since rules vary by federation, region, and rulebook version.

4. NO EVIDENCE FOUND: If the RETRIEVED CONTEXT does not contain information that answers the question, do not guess or fill the gap with general knowledge. Instead, clearly tell the user that you couldn't find a specific rule or evidence for their question in the rulebooks, phrased politely and in their language. Leave the "claims" list empty in this case.

5. CONVERSATIONAL CONTINUITY: Use the <chat_history> to understand previous context and continuity. When answering follow-up questions, speak naturally without repeating introductory pleasantries or re-explaining previously settled points unnecessarily.

6. OUT-OF-SCOPE QUESTIONS: If the question is clearly unrelated to sports rules (e.g. general knowledge, personal advice, coding help, politics), politely explain that you can only help with questions about the football, basketball, handball, tennis, and boxing rulebooks. Do not attempt to answer it. Leave the "claims" list empty.

7. TONE: Be warm, clear, and approachable — like a knowledgeable, friendly referee explaining a rule to someone, not a dry legal document. Avoid overly technical or robotic phrasing, while still being precise about the rule itself.

8. CITE YOUR CLAIMS: For every factual statement in your answer that comes from the rulebook, you must cite it by adding a corresponding entry to the "claims" list. To cite a claim means to:
   - Extract a specific, standalone factual statement (not the whole answer copy-pasted) that supports part of your answer
   - Attach the exact "file_name" and "page_number" from the context excerpt it came from, so the user can trace exactly where the rule was cited from
   - Only cite what is actually stated in the RETRIEVED CONTEXT — never cite a page number or rulebook name that wasn't provided to you
   - If your answer draws on multiple distinct facts, cite each one separately rather than bundling them into a single vague citation

9. MULTIPLE SPORTS IN CONTEXT: If the RETRIEVED CONTEXT contains excerpts from more than one sport's rulebook, use only the excerpts relevant to the sport the user is actually asking about. If the user's question is ambiguous about which sport they mean (e.g. "what's a foul?" could apply to several sports), ask a brief clarifying question instead of guessing, and leave "claims" empty.

10. DO NOT HALLUCINATE SOURCES: Never fabricate a rulebook name, page number, or quote that isn't explicitly present in the RETRIEVED CONTEXT above.
</instructions>

<constraints>
- Never answer a rules question using knowledge outside the RETRIEVED CONTEXT, even if you are confident it's correct — rulebooks can vary by edition/federation.
- Never mix rules from different sports into a single answer unless the user explicitly asks for a comparison.
- Never claim a rule exists if the RETRIEVED CONTEXT does not explicitly support it.
- Keep the "answer" field self-contained and understandable on its own, without requiring the reader to look at the "claims" field to understand it.
- If the RETRIEVED CONTEXT is empty or irrelevant to the question, do not populate the "claims" list — state clearly that no rule/evidence was found.
- Output must strictly follow the required JSON schema — do not add extra fields, commentary, or markdown formatting outside the schema.
</constraints>

<output_format>
Respond with a JSON object matching this exact structure:
{{
  "answer": "<final answer to the user, in their language>",
  "claims": [
    {{
      "claim": "<specific factual statement from the RETRIEVED CONTEXT>",
      "source": {{
        "file_name": "<rulebook name from context>",
        "page_number": "<page number from context>"
      }}
    }}
  ]
}}
</output_format>
"""

sports_prompt = ChatPromptTemplate.from_messages([
    ("system", sports_reference_system_prompt),
    ("human", "User Query / Situation: {query}")
])

In [72]:
import os
from langchain_groq import ChatGroq

# Recommended multilingual model on Groq
llm = ChatGroq(
    # model="openai/gpt-oss-120b",
    model="qwen/qwen3.8-27b",
    temperature=0.0,
    api_key=os.environ.get("GROQ_API_KEY")
)

# Bind structured schema
structured_llm = llm.with_structured_output(RefereeAnswer)
sports_rule_chain = sports_prompt | structured_llm


#### query parsing

In [73]:
from pydantic import BaseModel, Field
from typing import List, Optional

class SubQuestion(BaseModel):
    question: str = Field(
        description="Atomic, self-contained sub-question in English."
    )
    sport: Optional[str] = Field(
        default=None,
        description="The specific sport this question targets ('Football', 'Basketball', 'Handball', 'Tennis', 'Boxing'). If the question is generic across multiple sports, cannot be specified, or is not about sports, set to None."
    )

class QueryAnalysis(BaseModel):
    detected_language: str = Field(description="The language of the original user query, e.g. 'Arabic', 'English', 'French'")
    translated_query: str = Field(description="The full original query translated into English, preserving its complete meaning and intent")
    sub_questions: List[SubQuestion] = Field(
        description="Atomic, self-contained sub-questions in English with their specified sport (or None if generic/unspecified)."
    )


In [74]:
DECOMPOSER_SYSTEM_PROMPT = """
<role>
You are a query analysis assistant for a multi-sport rulebook RAG retrieval system covering exactly five sports: Football, Basketball, Handball, Tennis, and Boxing. Your job is NOT to answer the user's question — it is to prepare their query for retrieval by translating it, resolving conversational context/pronouns from chat history, and breaking it down into clear, atomic, self-contained sub-questions with their target sport.
</role>

<chat_history>
{chat_history}
</chat_history>

<task>
Given a user's latest query (which may be in any language) and any previous <chat_history>, perform these steps in order:

1. DETECT the language of the original query.

2. TRANSLATE the query into English. Preserve the complete meaning, intent, and terminology (e.g. "التسلل" -> "offside").

3. CONTEXT & COREFERENCE RESOLUTION:
   - Check <chat_history> to resolve pronouns (e.g. "he", "they", "this rule", "it"), elliptical follow-ups (e.g. "what if he was in his own half?", "why?"), or implied sports.
   - Formulate every sub-question so that it is COMPLETELY SELF-CONTAINED and can be understood and searched independently without needing to read the chat history.
   - If the user asks a follow-up about a previous sport (e.g., "what about handball?"), adapt the previous topic to the new sport.

4. CLASSIFY and DECOMPOSE the query into a list of SubQuestion objects, each having `question` (str) and `sport` (str or null):

   A) NOT ABOUT SPORT (greeting, chitchat, unrelated topic):
      Pass the ORIGINAL query through completely unchanged in original language with sport = null.

   B) CLEARLY ABOUT ONE SPECIFIC SPORT (stated or unambiguously resolved from history):
      Produce atomic English sub-question(s). Explicitly set `sport` to exactly one of ['Football', 'Basketball', 'Handball', 'Tennis', 'Boxing'].

   C) ABOUT SPORT RULES BUT SPORT IS UNKNOWN/GENERIC (could apply to multiple sports):
      Create ONE separate SubQuestion per plausible sport, with that sport's name in `sport` and in the `question` text.
      Example: "what counts as a foul?" -> create separate SubQuestions, each setting sport to Football, Basketball, Handball, Boxing respectively.

   D) ABOUT SPORT RULES, SPORT IS CLEAR, BUT QUERY IS VAGUE:
      Clarify the question in English and specify the `sport`.

5. If a sport cannot be specified or the question is generic/unrelated, set `sport` to null (None).
</task>

<examples>
Chat History:
User: "What is offside in football?"
Assistant: "A player is in an offside position if they are nearer to the opponents' goal line..."
Latest Query: "ماذا لو كان اللاعب في نصف ملعبه؟"
detected_language: "Arabic"
translated_query: "What if the player was in his own half?"
sub_questions: [
  {{"question": "Is a player penalized for offside if they are in their own half of the field of play in football?", "sport": "Football"}}
]

Chat History:
User: "What is offside in football?"
Assistant: "A player is in an offside position..."
Latest Query: "وهل يوجد هذا القانون في كرة اليد؟"
detected_language: "Arabic"
translated_query: "Does this rule exist in handball?"
sub_questions: [
  {{"question": "Does the offside rule exist in handball?", "sport": "Handball"}}
]

Original (Arabic, greeting): "مرحبا، كيف حالك؟"
detected_language: "Arabic"
translated_query: "Hello, how are you?"
sub_questions: [
  {{"question": "مرحبا، كيف حالك؟", "sport": null}}
]
</examples>

<constraints>
- Never answer the question — only translate, resolve conversational context, classify, and decompose it.
- Sport MUST be either null or exactly one of: 'Football', 'Basketball', 'Handball', 'Tennis', 'Boxing'.
- Output must strictly follow the QueryAnalysis schema.
</constraints>
"""

In [75]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
import asyncio

decomposer_llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
structured_decomposer = decomposer_llm.with_structured_output(QueryAnalysis)

decomposer_prompt = ChatPromptTemplate.from_messages([
    ("system", DECOMPOSER_SYSTEM_PROMPT),
    ("human", "{query}")
])

decomposer_chain = decomposer_prompt | structured_decomposer

def format_chat_history(chat_history: list[dict] | None = None) -> str:
    """Formats chat history list into clean readable text for prompts."""
    if not chat_history:
        return "No previous conversation history."
    formatted = []
    for msg in chat_history:
        role = "User" if msg.get("role") == "user" else "Assistant"
        content = msg.get("content", "")
        formatted.append(f"{role}: {content}")
    return "\n".join(formatted)

async def retrieve_with_decomposition(user_query: str, chat_history: list[dict] | None = None):
    """
    Decomposes the query using conversational context and retrieves parent docs
    for all sub-questions in parallel.
    """
    formatted_history = format_chat_history(chat_history)
    analysis = await decomposer_chain.ainvoke({
        "query": user_query,
        "chat_history": formatted_history
    })

    print(f"Detected language: {analysis.detected_language}")
    print(f"Translated query: {analysis.translated_query}")
    print(f"Sub-questions: {[(sq.question, sq.sport) for sq in analysis.sub_questions]}")

    # Parallel retrieval across all sub-questions using asyncio.to_thread
    tasks = [
        asyncio.to_thread(advanced_parent_child_retrieval, query=sub_q.question, sport=sub_q.sport)
        for sub_q in analysis.sub_questions
    ]
    results = await asyncio.gather(*tasks)

    all_docs = []
    for docs in results:
        all_docs.extend(docs)

    # Dedupe by doc_id
    seen = set()
    unique_docs = []
    for doc in all_docs:
        doc_id = doc.metadata.get('doc_id')
        if doc_id not in seen:
            seen.add(doc_id)
            unique_docs.append(doc)

    return unique_docs, analysis

In [76]:
async def ask_sports_rules(
    query: str, 
    chat_history: list[dict] | None = None, 
    return_sources: bool = False
):
    """
    Takes a user query and optional chat_history, decomposes the query using conversational context,
    retrieves parent documents in parallel via sport-filtered hybrid retrieval,
    and returns the grounded answer from the LLM.
    """
    # Step 1: Retrieve parent docs via Advanced Retriever with decomposition and history
    retrieved_docs, analysis = await retrieve_with_decomposition(query, chat_history=chat_history)
    
    if not retrieved_docs:
        msg = "No relevant rulebook sections found for this query."
        if return_sources:
            return {"answer": msg, "claims": [], "sources": [], "analysis": analysis}
        return RefereeAnswer(answer=msg, claims=[])
    
    # Step 2: Format context with source metadata
    formatted_context = "\n\n---\n\n".join([
        f"[{doc.metadata.get('file_name', 'Rulebook')} | Page {doc.metadata.get('page_number', 'N/A')}]:\n{doc.page_content}"
        for doc in retrieved_docs
    ])
    formatted_history = format_chat_history(chat_history)
    
    # Step 3: Generate grounded answer with chat history awareness
    response = sports_rule_chain.invoke({
        "query": query,
        "context": formatted_context,
        "chat_history": formatted_history
    })
    
    if return_sources:
        return {
            "answer": response.answer,      
            "claims": response.claims,      
            "sources": retrieved_docs
        }
    
    return response

class SportsChatSession:
    """
    Stateful chat session that manages conversational memory automatically across turns.
    """
    def __init__(self):
        self.history: list[dict] = []
        
    async def chat(self, user_query: str, return_sources: bool = True):
        res = await ask_sports_rules(user_query, chat_history=self.history, return_sources=return_sources)
        answer_text = res["answer"] if isinstance(res, dict) else (res.answer if hasattr(res, "answer") else str(res))
        
        self.history.append({"role": "user", "content": user_query})
        self.history.append({"role": "assistant", "content": answer_text})
        return res
        
    def reset(self):
        self.history.clear()
        print("Chat history reset.")

In [77]:
# =========================================================================
# Multi-turn Conversational Memory Demo
# =========================================================================
session = SportsChatSession()

# Turn 1: Initial Question
print("=== TURN 1 ===")
res1 = await session.chat("ما هو حكم التسلل في كرة القدم؟")
print("Referee Answer:\n", res1["answer"])

# Turn 2: Follow-up question with pronoun and context dependency
print("\n=== TURN 2 (Follow-up) ===")
res2 = await session.chat("وماذا لو كان اللاعب في نصف ملعبه الخاص؟")
print("Referee Answer:\n", res2["answer"])

# Turn 3: Cross-sport follow-up
print("\n=== TURN 3 (Cross-sport) ===")
res3 = await session.chat("وهل يوجد هذا القانون في كرة اليد؟")
print("Referee Answer:\n", res3["answer"])

=== TURN 1 ===
Detected language: Arabic
Translated query: What is the ruling of offside in football?
Sub-questions: [('What is the ruling of offside in football?', 'Football')]
Referee Answer:
 حكم التسلل في كرة القدم ينقسم إلى وضع التسلل والارتكاب الفعلي للمخالفة:\n\n1) وضع التسلل: لا يُعدّ وجود اللاعب في وضع تسلل بحد ذاته مخالفة. يُعتبر اللاعب في وضع تسلل إذا كان أي جزء من رأسه أو جسمه أو قدميه في نصف ملعب الخصم (باستثناء خط المنتصف)، وكان أقرب إلى خط مرمى الخصم من الكرة ومن ثاني آخر لاعب من الفريق الخصم. ولا يُحتسب وضع تسلل إذا كان اللاعب على مستوى ثاني آخر خصم أو على مستوى آخر لاعبين اثنين.\n\n2) مخالفة التسلل: يُعاقب اللاعب الذي يكون في وضع تسلل لحظة لعب الكرة أو لمسها من زميله إذا تورط في اللعب الفعلي من خلال: (أ) التدخل في اللعب بلمس الكرة أو لعبها بعد أن مررها أو لمسها زميله، أو (ب) التدخل في أداء الخصم.\n\n3) حالات لا تُعدّ مخالفة: لا توجد مخالفة تسلل إذا استلم اللاعب الكرة مباشرة من ركلة مرمى، أو رمية تماس، أو ركلة ركنية.\n\n4) العقوبة: إذا ارتُكبت مخالفة التسلل، يُمنح الفري

In [80]:
session.history

[{'role': 'user', 'content': 'ما هو حكم التسلل في كرة القدم؟'},
 {'role': 'assistant',
  'content': 'حكم التسلل في كرة القدم ينقسم إلى وضع التسلل والارتكاب الفعلي للمخالفة:\\n\\n1) وضع التسلل: لا يُعدّ وجود اللاعب في وضع تسلل بحد ذاته مخالفة. يُعتبر اللاعب في وضع تسلل إذا كان أي جزء من رأسه أو جسمه أو قدميه في نصف ملعب الخصم (باستثناء خط المنتصف)، وكان أقرب إلى خط مرمى الخصم من الكرة ومن ثاني آخر لاعب من الفريق الخصم. ولا يُحتسب وضع تسلل إذا كان اللاعب على مستوى ثاني آخر خصم أو على مستوى آخر لاعبين اثنين.\\n\\n2) مخالفة التسلل: يُعاقب اللاعب الذي يكون في وضع تسلل لحظة لعب الكرة أو لمسها من زميله إذا تورط في اللعب الفعلي من خلال: (أ) التدخل في اللعب بلمس الكرة أو لعبها بعد أن مررها أو لمسها زميله، أو (ب) التدخل في أداء الخصم.\\n\\n3) حالات لا تُعدّ مخالفة: لا توجد مخالفة تسلل إذا استلم اللاعب الكرة مباشرة من ركلة مرمى، أو رمية تماس، أو ركلة ركنية.\\n\\n4) العقوبة: إذا ارتُكبت مخالفة التسلل، يُمنح الفريق الخصم ركلة حرة غير مباشرة من مكان وقوع المخالفة، حتى لو كانت في نصف ملعب اللاعب المخ

In [83]:
session2 = SportsChatSession()

# Turn 1: Initial Question
print("=== TURN 1 ===")
res1 = await session2.chat("ما هو حكم التسلل في كرة القدم؟")
print("Referee Answer:\n", res1["answer"])

# Turn 2: Follow-up question with pronoun and context dependency
print("\n=== TURN 2 (Follow-up) ===")
res2 = await session2.chat("وماذا لو كان اللاعب في نصف ملعبه الخاص؟")
print("Referee Answer:\n", res2["answer"])

# Turn 3: Cross-sport follow-up
print("\n=== TURN 3 (Cross-sport) ===")
res3 = await session2.chat("What's the rule we're talking about")
print("Referee Answer:\n", res3["answer"])

=== TURN 1 ===
Detected language: Arabic
Translated query: What is the ruling on offside in football?
Sub-questions: [('What is the ruling on offside in football?', 'Football')]
Referee Answer:
 حكم التسلل في كرة القدم ينقسم إلى وضع التسلل والارتكاب الفعلي للمخالفة:\n\n1) وضع التسلل: لا يُعدّ مجرد وجود اللاعب في وضع تسلل مخالفة. يُعتبر اللاعب في وضع تسلل إذا كان أي جزء من رأسه أو جسمه أو قدميه في نصف الملعب الخاص بالخصم (باستثناء خط المنتصف)، وكان أقرب إلى خط مرمى الخصم من الكرة ومن ثاني آخر خصم. ولا تُحتسب الأيدي والأذرع (بما فيها حارس المرمى) في هذا الحكم. كما لا يكون اللاعب في وضع تسلل إذا كان على مستوى ثاني آخر خصم أو على مستوى آخر خصمين.\n\n2) مخالفة التسلل: يُعاقَب اللاعب الذي يكون في وضع تسلل لحظة لعب الكرة أو لمسها من زميله إذا شارك بنشاط في اللعب، وذلك من خلال: التدخل في اللعب بلمس الكرة أو لعبها بعد تمريرها أو لمسها من زميله، أو التدخل في منافسة الخصم، أو الحصول على ميزة من وضعه التسللي.\n\n3) لا توجد مخالفة تسلل إذا استلم اللاعب الكرة مباشرة من: ركلة مرمى، ركلة ركنية، أو رمي

In [1]:
from rag_pipeline import ask_sports_rules_sync
from rag_pipeline import ask_sports_rules


In [8]:
result = await ask_sports_rules("what's the offside",return_sources=True)

In [12]:
result["answer"]

'In football, being in an offside position is not an offence in itself. A player is in an offside position if any part of their head, body, or feet is in the opponents’ half (excluding the halfway line) and is nearer to the opponents’ goal line than both the ball and the second-last opponent. The hands and arms are not considered for this. A player is not in an offside position if they are level with the second-last opponent or the last two opponents.\n\nA player in an offside position is only penalised if they become involved in active play at the moment the ball is played or touched by a team-mate. This includes interfering with play by playing or touching the ball, or interfering with an opponent. There is no offside offence if a player receives the ball directly from a goal kick, a throw-in, or a corner kick. If an offside offence occurs, the referee awards an indirect free kick where the offence occurred.'